# MP2: Data Cleaning & Feature Engineering

**MajsterPlus — Customer Lapse Prediction**

In this mini-project, you will follow a **mandatory 12-step preprocessing pipeline**
to clean the data and prepare it for modeling. The steps must be executed in order.

**CRISP-DM Phase**: Data Preparation

---

## 0. Setup & Reproducibility

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Library version assertions
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — expected 1.4.x or 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — expected 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — expected <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## Step 1–2: Load Data & Verify Fingerprint

In [ ]:
import hashlib
from pathlib import Path

# Colab detection
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
except ImportError:
    DATA_DIR = Path("../2. data")

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

# Load and verify customers.csv
customers = pd.read_csv(DATA_DIR / "customers.csv")
cust_md5 = hashlib.md5((DATA_DIR / "customers.csv").read_bytes()).hexdigest()
assert cust_md5 == "c016cd4bfc36ac8d0a99bb6a3d55fa44", (
    f"customers.csv MD5 mismatch: {cust_md5} != c016cd4bfc36ac8d0a99bb6a3d55fa44"
)
print(f"customers.csv loaded: {customers.shape}, MD5 OK")

# Load and verify transactions.csv
txn_md5 = hashlib.md5((DATA_DIR / "transactions.csv").read_bytes()).hexdigest()
assert txn_md5 == "a9c4bcfc4878baae6f5c250d4d15d737", (
    f"transactions.csv MD5 mismatch: {txn_md5} != a9c4bcfc4878baae6f5c250d4d15d737"
)
print(f"transactions.csv verified: MD5 OK")

## Step 3: Separate Target & Drop ID

In [ ]:
# Save target before any transformations
y = customers["is_lapsed"].copy()
print(f"Target saved: {y.shape[0]} values, lapse rate = {y.mean():.3f}")

# Drop customer_id (not a feature) and is_lapsed (target)
# Keep gender for later fairness analysis
gender_series = customers["gender"].copy()

df = customers.drop(columns=["customer_id", "is_lapsed"])
print(f"Working DataFrame: {df.shape}")

## Step 4: Parse Polish Dates

The `registration_date` column uses Polish month abbreviations (e.g., "15-sty-2024").
We provide the parsing helper — your task is to apply it and verify the derived feature.

In [ ]:
# Helper: parse Polish date strings into datetime
POLISH_MONTHS = {
    "sty": 1, "lut": 2, "mar": 3, "kwi": 4, "maj": 5, "cze": 6,
    "lip": 7, "sie": 8, "wrz": 9, "pa\u017a": 10, "lis": 11, "gru": 12,
}

def parse_polish_date(date_str: str) -> pd.Timestamp:
    """Convert Polish date string 'DD-mmm-YYYY' to Timestamp."""
    day, month_str, year = date_str.split("-")
    return pd.Timestamp(year=int(year), month=POLISH_MONTHS[month_str], day=int(day))

In [ ]:
# TODO: Apply the parse_polish_date helper to the registration_date column.
# Use the reference date 2025-01-15 to derive account age in days.
# Verify that your derived values match the existing account_age_days column.
# Then drop registration_date (it is now redundant).

# YOUR CODE HERE


## Step 5: Clean total_spend

The `total_spend` column contains currency-formatted strings like `"PLN 1,234.50"`.
Convert it to a numeric float column.

In [ ]:
# TODO: Convert total_spend from currency string format ("PLN 1,234.50") to a numeric float.
# After conversion, print df["total_spend"].describe() to verify.

# YOUR CODE HERE


### Interpretation: total_spend distribution

**TODO**: Compare the mean and median of `total_spend` (use the `.describe()` output above). What does the discrepancy between these two measures tell you about the distribution shape? Is it right-skewed, left-skewed, or approximately normal? Why might this matter for modeling?

In [ ]:
# TODO: Print the mean and median of total_spend explicitly.
# Write your interpretation as a comment or print statement below.

# YOUR CODE HERE


## Step 6: Handle Impossible satisfaction_score Values

Valid range for `satisfaction_score` is [1.0, 5.0]. Any values outside this range
are data entry errors and should be replaced with NaN.

### Think About This

Why must impossible values be replaced with NaN **before** running imputation in Step 7? What would happen to the median if the impossible values (e.g., 0.0, 7.2, -1.0) were included in the computation?

In [ ]:
# TODO: Identify satisfaction_score values outside the valid range [1.0, 5.0]
# and replace them with NaN. Print how many impossible values you found.

# YOUR CODE HERE


## Step 7: Impute Missing Values

After Steps 4–6, several columns have missing values. Apply appropriate imputation:
- **Numeric columns** (age, online_ratio, satisfaction_score): median imputation
- **Categorical columns** (monthly_income_bracket, referral_source): mode imputation

In [ ]:
# TODO: Impute all missing values.
# For numeric columns use the median; for categorical columns use the mode.
# Print missing value counts before and after imputation.
# Remember: age was loaded as Int64 (nullable int); convert to regular int after imputation.

# YOUR CODE HERE


## Step 8: Remove IQR Outliers

The `avg_basket_value` column has extreme outliers. Use the IQR method
with factor=1.5 to identify and remove them.

In [ ]:
# IQR calculation (pre-filled)
Q1 = df["avg_basket_value"].quantile(0.25)
Q3 = df["avg_basket_value"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

In [ ]:
# TODO: Filter out rows where avg_basket_value is outside the IQR bounds.
# IMPORTANT: You must also filter y and gender_series using the same boolean mask.
# Why? Think about what happens if feature rows and target values become misaligned.
# Print how many rows were removed.

# YOUR CODE HERE


## Step 9: Encode Categorical Variables

Encode all categorical columns in the following order:
1. **Binary**: `loyalty_member` (Tak/Nie)
2. **Ordinal**: `monthly_income_bracket` (A through E, representing increasing income)
3. **One-hot**: remaining categorical columns with `pd.get_dummies(drop_first=False)`
4. Sort columns alphabetically and convert any boolean dummy columns to int.

In [ ]:
# TODO: Encode all categorical variables following the order above.
# After encoding, sort columns alphabetically: df = df[sorted(df.columns)]
# Print df.shape and df.dtypes to verify.

# YOUR CODE HERE


### Interpretation: drop_first trade-off

**TODO**: The course uses `drop_first=False`. What would happen if you used `drop_first=True` instead? Which types of models would benefit from dropping the first category, and why? Which types of models would be unaffected?

In [ ]:
# TODO: Write your answer as a comment or print statement.
# Consider: What is multicollinearity? Which models are sensitive to it?

# YOUR ANSWER HERE


## Step 10: Null Assertion Gate

In [ ]:
# This MUST pass before proceeding
assert df.isnull().sum().sum() == 0, (
    f"Still have {df.isnull().sum().sum()} null values! Fix Steps 6-7."
)
print(f"Null assertion passed. DataFrame shape: {df.shape}")
print(f"All dtypes numeric: {all(df.dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")

## Step 11: Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train lapse rate: {y_train.mean():.3f}")
print(f"y_test  lapse rate: {y_test.mean():.3f}")

## Step 12: Feature Scaling (StandardScaler)

Fit a `StandardScaler` on the training set and transform both train and test sets.
Remember: fit on train only, transform both — fitting on full data is data leakage.

In [ ]:
# TODO: Apply StandardScaler. Fit on training data ONLY, then transform both sets.
# Preserve the DataFrame structure (columns and index) in the scaled output.

from sklearn.preprocessing import StandardScaler

# YOUR CODE HERE


## Bonus: K-Means Clustering

**Optional**: Use K-Means to create customer segments as an additional feature.
Use 3 clusters on scaled behavioral features (purchase_count, avg_basket_value,
days_since_last_purchase, product_categories_bought).

**Task**: Fit K-Means, add cluster labels as a feature to both train and test.

In [ ]:
# YOUR CODE HERE (optional bonus)
# from sklearn.cluster import KMeans
# Fit on behavioral features from training set
# Add cluster labels to both X_train_scaled and X_test_scaled

## Save Checkpoint for MP3

In [ ]:
import pickle

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Split gender_series to align with train/test
gender_train = gender_series.loc[X_train.index]
gender_test = gender_series.loc[X_test.index]

checkpoint = {
    "X_train": X_train_scaled if "X_train_scaled" in dir() else X_train,
    "X_test": X_test_scaled if "X_test_scaled" in dir() else X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_names": list(df.columns),
    "gender_train": gender_train,
    "gender_test": gender_test,
}

with open(CHECKPOINT_DIR / "mp2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)
print(f"Checkpoint saved: {CHECKPOINT_DIR / 'mp2_checkpoint.pkl'}")
print(f"Keys: {list(checkpoint.keys())}")

---
*End of MP2 — Continue to MP3: Baseline Modeling & Algorithm Comparison*